# Understanding Generative Adversarial Networks (GANs) & DCGANs

**Generative Adversarial Networks (GANs)** are a class of artificial intelligence architectures that learn to create new, realistic data (such as images, sounds, or text) through internal competition. Instead of explicitly programming the network on how to draw a specific object, GANs rely on two separate neural networks playing a competitive "zero-sum game."

### The Core Architecture: A Game of Cat and Mouse
* **The Generator (The Counterfeiter):** Its goal is to create fake data from random noise. It aims to produce samples so realistic that they are indistinguishable from real data.
* **The Discriminator (The Police):** Its job is to evaluate the data. It looks at both real samples from the dataset and fake samples from the Generator, trying to correctly classify them as real or fake.

### The Mathematical Intuition: Landscapes and Tug-of-War
While we can describe this as a game of cops and robbers, the underlying math is about **probability distributions**. 

Imagine all possible images in the world exist on a vast, invisible map. The *real* images we want to generate (like human faces) are clustered together, forming high "mountains" or peaks on this map. Everything else (like random TV static) lies in the flat valleys. 
* **Matching the Landscape:** The Generator starts by blindly dropping random points (noise) into the valleys. Its ultimate goal is to mold its own generated landscape until it perfectly overlaps the mountain range of the real data. 
* **Measuring the Distance:** The Discriminator acts as a surveyor. It calculates the "distance" or difference between the real mountains and the Generator's fake hills. 
* **The Minimax Game:** This is a mathematical tug-of-war. The Discriminator is pulling the rope to *maximize* the measurable distance between real and fake (getting better at spotting the fakes). Simultaneously, the Generator pulls the rope in the exact opposite direction, trying to *minimize* that distance to zero. When the distance becomes zero, the landscapes overlap perfectly, and we reach a "Nash Equilibrium" where the Discriminator can only guess 50/50.

### The Evolution: DCGAN (Deep Convolutional GAN)
Early GANs were notoriously unstable because finding that perfect equilibrium is incredibly difficult. **DCGANs** solved many of these issues by replacing standard dense layers with Deep Convolutional Neural Networks (CNNs). Key architectural improvements include:
* **Strided Convolutions:** Replacing standard pooling layers to let the network learn spatial downsampling and upsampling on its own.
* **Batch Normalization:** Stabilizing the learning process and preventing the model from oscillating wildly.
* **No Fully Connected Layers:** Relying on a fully convolutional architecture for better spatial understanding.
* **Optimized Activations:** Using ReLU in the Generator and LeakyReLU in the Discriminator to prevent "dying" neurons.

### Common Training Pitfalls
While the underlying math is elegant, real-world GAN training is highly volatile. Common issues include:
* **Vanishing Gradients (The "Optimal" Discriminator):** If the Discriminator surveyor becomes too good too quickly, the "distance" it measures becomes absolute. It stops providing useful directional feedback (gradients) to the Generator, completely halting the learning process.
* **Non-convergence:** Because two networks are pulling the rope in opposite directions, they might end up in an endless loop, constantly overcorrecting without ever settling on the overlapping landscapes.
* **Mode Collapse:** A notorious issue where the Generator figures out how to build just *one* highly realistic peak that consistently fools the Discriminator. Instead of mapping the entire mountain range (generating diverse faces), it safely stays on that single peak, producing the exact same image over and over.

**Great video on GANs**: https://www.youtube.com/watch?v=RAa55G-oEuk

---
# Building DCGAN to generate images, based on Intel Image dataset from Kaggle

In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torch_directml

In [ ]:
BATCH_SIZE = 128
IMAGE_SIZE = 64
CHANNELS = 3
DEVICE = torch_directml.device(0)
LATENT_DIM = 100
GEN_FEATURES = 64
DISC_FEATURES = 64
LEARNING_RATE = 0.0002
BETA1 = 0.5 
REAL_LABEL = 1.
FAKE_LABEL = 0.
EPOCHS = 100

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    # Normalize the images to [-1, 1] range
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
dataset = torchvision.datasets.ImageFolder(root='./data/intel-image/train', transform=transform)
dataloader = DataLoader(dataset=dataset, batch_size=BATCH_SIZE, shuffle=True)

### DCGAN Generator Architecture

The Generator is responsible for upsampling a random noise vector (latent space) into a fake 64x64 RGB image.

**Core DCGAN Best Practices Implemented:**
* **No Fully Connected Layers:** Uses only fractional-strided convolutions (`ConvTranspose2d`) to learn spatial upsampling.
* **Progressive Scaling:** Each block halves the number of feature maps while doubling the spatial dimensions (4x4 $\rightarrow$ 8x8 $\rightarrow$ 16x16 $\rightarrow$ 32x32 $\rightarrow$ 64x64).
* **`bias=False`:** Bias terms are disabled in convolutional layers because the subsequent `BatchNorm2d` layer effectively neutralizes and replaces them, saving memory.
* **Batch Normalization:** Applied to all hidden layers to stabilize training and ensure healthy gradient flow. 
* **Hidden Activations:** `ReLU` is used in all hidden layers.
* **Output Layer:** No Batch Normalization is applied here. Uses the `Tanh` activation function to map the final pixel values to the $[-1, 1]$ range, matching the normalized real images.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.main = nn.Sequential(
            # ---Block 1: From noise (1x1) to small image (4x4)---
            nn.ConvTranspose2d(in_channels=LATENT_DIM, out_channels=GEN_FEATURES * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(num_features=GEN_FEATURES * 8),
            nn.ReLU(inplace=True),
            # Output: [512, 4, 4]

            # ---Block 2: From 4x4 to 8x8---
            nn.ConvTranspose2d(in_channels=GEN_FEATURES * 8, out_channels=GEN_FEATURES * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_features=GEN_FEATURES * 4),
            nn.ReLU(inplace=True),
            # Output: [256, 8, 8]

            # ---Block 3: From 8x8 to 16x16---
            nn.ConvTranspose2d(in_channels=GEN_FEATURES * 4, out_channels=GEN_FEATURES * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_features=GEN_FEATURES * 2),
            nn.ReLU(inplace=True),
            # Output: [128, 16, 16]

            # ---Block 4: From 16x16 to 32x32---
            nn.ConvTranspose2d(in_channels=GEN_FEATURES * 2, out_channels=GEN_FEATURES, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_features=GEN_FEATURES),
            nn.ReLU(inplace=True),
            # Output: [64, 32, 32]

            # ---Block 5: From 32x32 to 64x64---
            nn.ConvTranspose2d(in_channels=GEN_FEATURES, out_channels=CHANNELS, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
            # Output: [3, 64, 64]
        )

    def forward(self, x):
        return self.main(x)


In [ ]:
netG = Generator().to(DEVICE)

In [ ]:
noise = torch.randn(1, LATENT_DIM, 1, 1, device=DEVICE)
test_output = netG(noise)
test_output.shape

### DCGAN Discriminator Architecture

The Discriminator acts as a standard binary classifier, taking a 64x64 RGB image and downsampling it to output a single probability score (1 for real, 0 for fake).

**Core DCGAN Best Practices Implemented:**
* **No Pooling Layers:** Replaces standard max-pooling with strided convolutions (`stride=2`) to allow the network to learn its own spatial downsampling.
* **Input Layer:** Crucially, **no Batch Normalization** is applied to the very first layer (Block 1). This ensures the model learns the true data distribution directly from the raw input images.
* **Hidden Activations:** `LeakyReLU` (with a negative slope of 0.2) is used instead of standard `ReLU`. This prevents the "dying ReLU" problem and ensures gradients continue to flow backward to train the Generator, even for negative values.
* **Output Layer:** The final convolution uses `stride=1` and `padding=0` to collapse the 4x4 spatial dimensions into a single scalar value. A `Sigmoid` activation function is then applied to constrain the final output to a valid probability range of $[0, 1]$.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.main = nn.Sequential(
            #---Block 1: From 64x64 to 32x32---
            nn.Conv2d(in_channels=CHANNELS, out_channels=DISC_FEATURES, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # Output: [64, 32, 32]

            #---Block 2: From 32x32 to 16x16---
            nn.Conv2d(in_channels=DISC_FEATURES, out_channels=DISC_FEATURES * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_features=DISC_FEATURES * 2),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # Output: [128, 16, 16]

            #---Block 3: From 16x16 to 8x8---
            nn.Conv2d(in_channels=DISC_FEATURES * 2, out_channels=DISC_FEATURES * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_features=DISC_FEATURES * 4),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # Output: [256, 8, 8]

            #---Block 4: From 8x8 to 4x4---
            nn.Conv2d(in_channels=DISC_FEATURES * 4, out_channels=DISC_FEATURES * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_features=DISC_FEATURES * 8),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            # Output: [512, 4, 4]

            #---Block 5: From 4x4 to 1x1---
            nn.Conv2d(in_channels=DISC_FEATURES * 8, out_channels=1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
            # Output: [1, 1, 1]
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)

In [ ]:
netD = Discriminator().to(DEVICE)

In [ ]:
# Test interaction between Generator and Discriminator
fake_image = netG(torch.randn(1, LATENT_DIM, 1, 1, device=DEVICE))
decision = netD(fake_image)
print(f"Decision from Discriminator for generated image: {decision.item()}")

### Custom Weight Initialization

In DCGANs, relying on the default PyTorch weight initialization often leads to training instability. The original paper dictates a specific custom initialization strategy.

**How the `weights_init` function works:**
* **Purpose:** It prevents gradient explosion or vanishing at the very start of training by keeping initial weights small and controlled.
* **Convolutional Layers (`Conv`):** Weights are randomly initialized from a Normal distribution with a mean of $0.0$ and a standard deviation of $0.02$ ($\mathcal{N}(0.0, 0.02)$).
* **Batch Normalization Layers (`BatchNorm`):** Weights are initialized from a Normal distribution with a mean of $1.0$ and a standard deviation of $0.02$ ($\mathcal{N}(1.0, 0.02)$). The bias terms are explicitly set to $0$.
* **Application:** This function is applied recursively to every layer before training begins using `netG.apply(weights_init)` and `netD.apply(weights_init)`.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__

    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### Optimizers: The `Betas` Parameter (Adam)

While not explicitly in the training loop snippet, defining the Adam optimizers requires tuning the `betas` parameter, specifically `beta1`.
*   **Standard Adam:** Typically uses a momentum parameter of `beta1 = 0.9`. This works well for static loss landscapes because it builds up speed in consistent directions.
*   **The GAN Problem:** In GANs, $G$ and $D$ are constantly fighting, meaning the loss landscape shifts wildly every iteration. High momentum causes the optimizer to "overshoot" the Nash Equilibrium, leading to severe instability, oscillations, or mode collapse.
*   **The DCGAN Solution:** `beta1` is lowered to `0.5` (`betas=(0.5, 0.999)`). This severely limits the momentum, forcing both networks to adapt more cautiously and stabilizing the training process.

In [ ]:
netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()

optimizerD = torch.optim.Adam(netD.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
optimizerG = torch.optim.Adam(netG.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

### The Training Loop (Minimax Game)

The training loop implements the competitive zero-sum game between the two networks.

**Phase 1: Training the Discriminator (The Police)**
*   **Goal:** Maximize the ability to correctly classify both real and fake images.
*   **Real Batch:** Pass real images through $D$ and calculate loss against `REAL_LABEL` (1).
*   **Fake Batch:** Generate fake images from random noise. 
*   **The `.detach()` Trick:** We use `fake_images.detach()` before passing them to $D$. This severs the computational graph, preventing PyTorch from calculating gradients for the Generator during the Discriminator's training phase. Calculate loss against `FAKE_LABEL` (0).
*   **Update:** Sum both errors (`errD_real + errD_fake`) and update $D$'s weights.

**Phase 2: Training the Generator (The Counterfeiter)**
*   **The "Non-Saturating Loss" Trick:** Theoretically, the Generator wants to minimize $\log(1 - D(G(z)))$. However, early in training, the Discriminator is much better than the Generator. This pushes the probability to 0 and kills the gradient (vanishing gradient). 
*   **The Solution:** Instead, we train the Generator to *maximize* $\log(D(G(z)))$. We do this practically by passing the fake images through $D$ (without detach, to calculate $G$'s gradients) and calculating the loss against `REAL_LABEL` (1). We are essentially penalizing the Generator for failing to output a "Real" classification.

**Monitoring Progress (`fixed_noise`)**
A static noise vector (`fixed_noise`) is created once before the loop. By passing this exact same noise through the Generator at the end of every epoch, we can visually track how the network's specific outputs evolve over time, avoiding the confusion of random variations.

In [ ]:
from IPython.display import clear_output

fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)

G_losses = []
D_losses = []

for epoch in range(EPOCHS):
    for i, data in enumerate(dataloader, 0):
        # --- Phase 1: Training the Discriminator ---
        # --- Goal: Maximize log(D(x)) + log(1 - D(G(z))) ---

        # --- Train with real images ---
        netD.zero_grad()

        real_images = data[0].to(DEVICE)
        b_size = real_images.size(0)

        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=DEVICE)

        output = netD(real_images)              # Discriminator's prediction for real images
        errD_real = criterion(output, label)    # Compute loss for real images
        errD_real.backward()                    # Calculate gradients for real images
        D_x = output.mean().item()              # Average output of the Discriminator for real images


        # --- Train with fake images ---
        noise = torch.randn(b_size, LATENT_DIM, 1, 1, device=DEVICE)  # Generate random noise
        fake_images = netG(noise)                                     # Generate fake images using the Generator

        label.fill_(FAKE_LABEL)     # Set labels for fake images

        # WATCH OUT FOR DETACH: We cut off fake_images from generator's graph to avoid backpropagating through the generator when training the discriminator.
        output = netD(fake_images.detach())

        errD_fake = criterion(output, label)   # Compute loss for fake images
        errD_fake.backward()                   # Calculate gradients for fake images
        D_G_z1 = output.mean().item()          # Average output of the Discriminator for fake images

        errD = errD_real + errD_fake           # Total loss for the Discriminator
        optimizerD.step()


        # --- Phase 2: Training the Generator ---
        # --- Goal (theoretical): Minimize log(1 - D(G(z))) ---
        # --- Goal (practical): Maximize log(D(G(z))) ---
        netG.zero_grad()

        # TRICK FOR GANs: We ask judge for error for 0 (fake), but we use the label 1 (real) as final answers
        # Why? Because generator wants discriminator to make a mistake and classify fake images as real. So we want to maximize the probability of discriminator being wrong.
        label.fill_(REAL_LABEL)

        output = netD(fake_images)              # Discriminator's prediction for fake images, without detach, because we want to calculate gradients for the generator

        errG = criterion(output, label)         # Compute loss for the Generator
        errG.backward()                         # Calculate gradients for the Generator
        D_G_z2 = output.mean().item()           # Average output of the Discriminator for fake images after Generator update
        optimizerG.step()

        # Save losses for plotting later
        G_losses.append(errG.item())
        D_losses.append(errD.item())

        if i % 100 == 0:
            print(f"[{epoch}/{EPOCHS}][{i}/{len(dataloader)}] Loss_D: {errD.item():.4f} Loss_G: {errG.item():.4f} D(x): {D_x:.4f} D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}")

    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()

    clear_output(wait=True)
    plt.imshow(np.transpose(torchvision.utils.make_grid(fake_test, padding=2, normalize=True), (1, 2, 0)))
    plt.show()